# TP1/TP1 Concurrent Analysis - Graphs

This file contains the scatterplot and model fit for the TP1/TP1 concurrent analysis of canonical proportion (CP) as a predictor of the following speech language measures:
- GFTA
- EVT

The weighted, scaled models are used on the scaled outcome measures.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from matplotlib.gridspec import GridSpec
from matplotlib.colors import to_rgba

# === Load ===
df = pd.read_csv("tp1_tp1_master_with_scaled.csv")
# recompute CP z-score over the full cohort (not the GFTA-complete subset)
df["canonical_proportion_scaled"] = (df["canonical_proportion"] - df["canonical_proportion"].mean()) / df["canonical_proportion"].std(ddof=0)

# --- Weights: num_clips = 0 + 1 (mean-normalized) ---
df["num_clips"] = pd.to_numeric(df["0"], errors="coerce").fillna(0) + \
                  pd.to_numeric(df["1"], errors="coerce").fillna(0)
w = df["num_clips"].astype(float).clip(lower=1)
w = w / w.mean()

# --- Gender dummy: 1 = Female, 0 = Male ---
def to_gender_dummy(x):
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Scale covariates (recommended for stability) ---
for col in ["age_mos", "Maternal_Education_Level"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

sc_age = StandardScaler()
sc_med = StandardScaler()
df["age_mos_scaled"] = sc_age.fit_transform(df[["age_mos"]])
df["Maternal_Education_Level_scaled"] = sc_med.fit_transform(df[["Maternal_Education_Level"]])

# canonical_proportion_scaled is assumed present in the CSV

# --- Common WLS fitter with controls; predictions at mean controls ---
def fit_weighted_with_controls(y_col, data, weights):
    predictors = [
        "canonical_proportion_scaled",
        "age_mos_scaled",
        "gender_dummy",
        "Maternal_Education_Level_scaled",
    ]
    needed = [y_col, "num_clips"] + predictors
    keep = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()

    X = sm.add_constant(keep[predictors])
    y = keep[y_col]
    ww = weights.loc[keep.index].to_numpy()

    model = sm.WLS(y, X, weights=ww).fit()

    # Grid over observed CP (global grid will be set outside)
    cp_min = keep["canonical_proportion_scaled"].min()
    cp_max = keep["canonical_proportion_scaled"].max()

    # Controls held at means
    ctrl_means = {
        "age_mos_scaled": float(keep["age_mos_scaled"].mean()),
        "gender_dummy": float(keep["gender_dummy"].mean()),  # proportion female
        "Maternal_Education_Level_scaled": float(keep["Maternal_Education_Level_scaled"].mean())
    }

    return {
        "model": model,
        "keep": keep,
        "cp_min": float(cp_min),
        "cp_max": float(cp_max),
        "ctrl_means": ctrl_means,
        "y_col": y_col
    }

# --- Outcomes (loop meta) ---
outcomes = [
    ("GFTA_Standard_scaled", "A", "GFTA-2 (SS)", "#1f77b4"),  # blue
    ("EVT_Standard_scaled",  "B", "EVT-2 (SS)",  "#ffbf00"),  # amber
]

# --- Fit all ---
fits = []
for y_col, tag, ylab, color in outcomes:
    if y_col not in df.columns:
        print(f"[skip] Missing column: {y_col}")
        fits.append(None)
    else:
        fits.append(fit_weighted_with_controls(y_col, df, w))

# --- Global CP grid so both panels span the same x-range ---
cp_all = df["canonical_proportion_scaled"].dropna()
x_lo, x_hi = float(cp_all.min()), float(cp_all.max())
cp_grid = np.linspace(x_lo, x_hi, 400)

# --- Figure like the multi-graph style (legend column on right) ---
plt.rcParams.update({
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})
fig = plt.figure(figsize=(13.2, 5.6), constrained_layout=False)
gs = GridSpec(
    nrows=1, ncols=3, figure=fig,
    width_ratios=[1, 1, 0.42],
    wspace=0.25, hspace=0.25
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
]
ax_leg = fig.add_subplot(gs[0, 2]); ax_leg.axis("off")

def plot_panel(ax, fit_obj, color, panel_tag, ylabel):
    if fit_obj is None:
        ax.set_visible(False); return

    model = fit_obj["model"]
    keep  = fit_obj["keep"]
    ctrl_means = fit_obj["ctrl_means"]
    y_col = fit_obj["y_col"]

    # Bubble size + alpha by clips
    sizes = np.clip(keep["num_clips"]/30.0, 20, 600)
    cmin, cmax = keep["num_clips"].min(), keep["num_clips"].max()
    denom = (cmax - cmin) if cmax > cmin else 1.0
    alphas = 0.3 + 0.7 * (keep["num_clips"] - cmin) / denom
    r, g, b, _ = to_rgba(color)
    facecols = [(r, g, b, float(a)) for a in alphas]

    # Scatter
    ax.scatter(keep["canonical_proportion_scaled"], keep[y_col],
               s=sizes, edgecolor="k", linewidth=0.35, c=facecols)

    # Predictions: controls held at means, CP on global grid
    Xp = pd.DataFrame({
        "const": 1.0,
        "canonical_proportion_scaled": cp_grid,
        "age_mos_scaled": ctrl_means["age_mos_scaled"],
        "gender_dummy": ctrl_means["gender_dummy"],
        "Maternal_Education_Level_scaled": ctrl_means["Maternal_Education_Level_scaled"],
    })[model.params.index]  # ensure correct order/labels

    pred = model.get_prediction(Xp).summary_frame(alpha=0.05)
    ax.plot(cp_grid, pred["mean"], color=color, linewidth=2.6)
    ax.fill_between(cp_grid, pred["mean_ci_lower"], pred["mean_ci_upper"],
                    color=color, alpha=0.25)

    # Labels & annotations
    ax.set_xlabel("Canonical Proportion (Scaled)")
    ax.set_ylabel(ylabel)
    ax.text(0.03, 0.92, f"({panel_tag})", transform=ax.transAxes,
            fontsize=16, fontweight="bold")
    if "canonical_proportion_scaled" in model.params.index:
        beta = model.params["canonical_proportion_scaled"]
        ax.text(0.05, 0.06, f"β = {beta:.2f}", transform=ax.transAxes,
                fontsize=20, fontweight="bold")

    # Limits: x spans full global CP; y by robust panel range
    ax.set_xlim(x_lo, x_hi)
    q1, q99 = keep[y_col].quantile([0.01, 0.99])
    y_pad = 0.10 * (q99 - q1 + 1e-9)
    ax.set_ylim(float(q1 - y_pad), float(q99 + y_pad))

    # Square panels + full borders
    ax.set_box_aspect(1)
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(1)

# --- Draw panels in a loop ---
for ax, fit_obj, meta in zip(axes, fits, outcomes):
    y_col, tag, ylab, color = meta
    plot_panel(ax, fit_obj, color, tag, ylab)

# --- Legend (clip sizes) on the right column ---
# clip_sizes = [50, 200, 1000, 3000, 5000, 8000]
# handles = [ax_leg.scatter([], [], s=np.clip(s/40.0, 20, 600), color="gray",
#                           alpha=0.7, edgecolor="k", linewidth=0.35)
#            for s in clip_sizes]
# labels = [f"= {s} clips" for s in clip_sizes]
# ax_leg.legend(handles, labels, loc="center left", frameon=True,
#               borderpad=1.6, labelspacing=1.6, handletextpad=1.2,
#               borderaxespad=0.6, fontsize=12, title="Clip Size (0 + 1)",
#               title_fontsize=12)

# Trim outer margins without re-introducing gaps
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.show()
